# Interview walkthrough — NBA home-win probability

Runs top-to-bottom. All modeling code is imported from `src/nba_wp` — nothing is re-implemented in cells.
Requires `data/nba-win-probability-data.csv` in the repo root's `data/` folder.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "configs" / "model.yaml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from nba_wp.config import load_config
from nba_wp.data import load_games, audit_games
from nba_wp.features import Architecture, build_features
from nba_wp.model import evaluate, fit_direct_logistic, predict_direct_logistic, predict_from_spec
from nba_wp.evaluation import baseline_probabilities, per_game_log_loss

cfg = load_config(ROOT / "configs" / "model.yaml")
games = load_games(ROOT / "data" / "nba-win-probability-data.csv")
spec = json.loads((ROOT / "artifacts/current/selected_spec_pre_march.json").read_text())
print(f"{len(games)} games loaded; selected spec: {spec['architecture']['name']}, C={spec['logistic_c']}")

1230 games loaded; selected spec: k5_hfa80_hl20, C=0.03


## 1. Data checks — pregame vs postgame columns

In [2]:
audit = audit_games(games)
print("home-win rate:", round(audit["home_win_rate"], 4))
print("record reconciliation mismatches:", audit["pregame_record_reconciliation"]["mismatch_count"])
print("missing:", audit["missing_value_count"], "| duplicate ids:", audit["duplicate_game_id_count"], "| ties:", audit["tied_game_count"])

home-win rate: 0.5545
record reconciliation mismatches: 0
missing: 0 | duplicate ids: 0 | ties: 0


## 2. Temporal split

In [3]:
counts = {
    "development (Oct-Feb)": int((games["game_date"] < "2026-03-01").sum()),
    "locked test (March)":   int(((games["game_date"] >= "2026-03-01") & (games["game_date"] < "2026-04-01")).sum()),
    "forecast (April)":      int((games["game_date"] >= "2026-04-01").sum()),
}
pd.Series(counts)

development (Oct-Feb)    895
locked test (March)      239
forecast (April)          96
dtype: int64

## 3. One team's state immediately before a game

In [4]:
architecture = Architecture.from_dict(spec["architecture"])
features = build_features(games, architecture)
example = features[features["game_date"] >= "2026-04-01"].iloc[0]
example[["game_id", "game_date", "home", "away", "elo_diff", "bt_logit", "trend_diff"]]

game_id                0022501003
game_date     2026-04-01 00:00:00
home                          MEM
away                          NYK
elo_diff                -0.293598
bt_logit                -0.689114
trend_diff              -6.049704
Name: 1134, dtype: object

## 4. A game's own result cannot change its own features

Flip the winner of that exact game and rebuild — the pregame feature row must be identical.

In [5]:
gid = example["game_id"]
flipped = games.copy()
row_mask = flipped["game_id"] == gid
flipped.loc[row_mask, ["home_points", "away_points"]] = flipped.loc[row_mask, ["away_points", "home_points"]].to_numpy()
flipped["home_win"] = (flipped["home_points"] > flipped["away_points"]).astype("int8")
features_flipped = build_features(flipped, architecture)
a = features.loc[features["game_id"] == gid, ["elo_diff", "bt_logit", "trend_diff"]].to_numpy()
b = features_flipped.loc[features_flipped["game_id"] == gid, ["elo_diff", "bt_logit", "trend_diff"]].to_numpy()
assert np.allclose(a, b, atol=0), "LEAK: same-game result changed its own features"
print("PASS: identical pregame features:", a.round(6))

PASS: identical pregame features: [[-0.293598 -0.689114 -6.049704]]


## 5. Baselines vs selected model (locked March, paired per-game log loss)

In [6]:
train_feb = features[features["game_date"] < "2026-03-01"]
march = features[(features["game_date"] >= "2026-03-01") & (features["game_date"] < "2026-04-01")]
probs = baseline_probabilities(train_feb, march, spec)
y = march["home_win"].to_numpy(dtype=int)
rows = []
sel_loss = per_game_log_loss(y, probs["selected_model"])
for name, p in probs.items():
    m = evaluate(march["home_win"], p)
    rows.append({"model": name, "log_loss": round(m["log_loss"], 4), "brier": round(m["brier"], 4),
                 "auc": round(m["auc"], 4), "accuracy": round(m["accuracy"], 4),
                 "mean_dLL_selected_minus_model": round(float((sel_loss - per_game_log_loss(y, p)).mean()), 4)})
pd.DataFrame(rows)

,model,log_loss,brier,auc,accuracy,mean_dLL_selected_minus_model
0,constant_home_rate,0.6806,0.2437,0.5000,0.6025,-0.1671
1,record_difference_logistic,0.5447,0.1800,0.8232,0.7741,-0.0312
2,elo_only_logistic,0.5039,0.1655,0.8241,0.7657,0.0095
3,selected_model,0.5135,0.1684,0.8241,0.7699,0.0000


## 6. Prequential pre-March selection (the search that picked the model)

In [7]:
sel_table = pd.read_csv(ROOT / "artifacts/current/pre_march_selection_results.csv")
top = sel_table.head(8)[["candidate", "feature_set", "logistic_c", "prequential_games",
                          "prequential_log_loss", "prequential_brier", "prequential_auc"]]
print(f"{len(sel_table)} candidates judged on pooled prequential Jan-Feb log loss; top 8:")
top.round(5)

672 candidates judged on pooled prequential Jan-Feb log loss; top 8:


,candidate,feature_set,logistic_c,prequential_games,prequential_log_loss,prequential_brier,prequential_auc
0,k5_hfa80_hl20|elo|C0.03,elo,0.03,399,0.63089,0.21962,0.70733
1,k5_hfa80_hl20|elo|C0.1,elo,0.10,399,0.63104,0.21929,0.70660
2,k5_hfa65_hl20|elo|C0.03,elo,0.03,399,0.63117,0.21973,0.70673
3,k5_hfa65_hl20|elo|C0.1,elo,0.10,399,0.63132,0.21939,0.70615
4,k5_hfa50_hl20|elo|C0.03,elo,0.03,399,0.63146,0.21985,0.70623
5,k5_hfa50_hl20|elo|C0.1,elo,0.10,399,0.63162,0.21951,0.70595
6,k5_hfa80_hl20|elo|C0.3,elo,0.30,399,0.63173,0.21938,0.70663
7,k5_hfa65_hl20|elo|C0.3,elo,0.30,399,0.63201,0.21948,0.70595


## 7. Locked March result (scored once)

In [8]:
final = json.loads((ROOT / "artifacts/current/final_metrics.json").read_text())
mm = final["locked_march_test"]["metrics"]
print(f"March: LL={mm['log_loss']:.4f} Brier={mm['brier']:.4f} AUC={mm['auc']:.4f} correct={mm['correct_games']}/239")

March: LL=0.5135 Brier=0.1684 AUC=0.8241 correct=184/239


## 8. Calibration (reliability of frozen April probabilities)

In [9]:
metrics = json.loads((ROOT / "reports/metrics.json").read_text())
pd.DataFrame(metrics["frozen_april_forecast"]["reliability_selected_model"])

,bin_low,bin_high,n,mean_prediction,observed_rate
0,0.0,0.1,1,0.087096,0.000000
1,0.1,0.2,10,0.176555,0.100000
2,0.2,0.3,8,0.253466,0.250000
3,0.3,0.4,14,0.352948,0.214286
4,0.4,0.5,11,0.461422,0.636364
5,0.5,0.6,10,0.562357,0.600000
6,0.6,0.7,10,0.658286,0.900000
7,0.7,0.8,9,0.745278,0.888889
8,0.8,0.9,17,0.848475,0.882353
9,0.9,1.0,6,0.921971,1.000000


## 9. Generate April predictions (frozen March 31 state)

In [10]:
frozen = build_features(games, architecture, freeze_date="2026-04-01")
train = frozen[frozen["game_date"] < "2026-04-01"]
april = frozen[frozen["game_date"] >= "2026-04-01"].copy()
p, _, _, _ = predict_from_spec(spec, train, april)
april["home_win_probability"] = p
m = evaluate(april["home_win"], p)
print(f"Frozen April: LL={m['log_loss']:.4f} Brier={m['brier']:.4f} AUC={m['auc']:.4f} correct={m['correct_games']}/96")

Frozen April: LL=0.4746 Brier=0.1530 AUC=0.8655 correct=75/96


## 10. Representative games with fair odds

In [11]:
view = april.sort_values("home_win_probability")[["game_id", "game_date", "away", "home", "home_win_probability"]]
sample = pd.concat([view.head(3), view.iloc[len(view)//2 - 1: len(view)//2 + 2], view.tail(3)])
sample = sample.assign(home_fair_odds=(1 / sample["home_win_probability"]).round(3),
                       away_fair_odds=(1 / (1 - sample["home_win_probability"])).round(3))
sample.round({"home_win_probability": 4})

,game_id,game_date,away,home,home_win_probability,home_fair_odds,away_fair_odds
1217,0022501188,2026-04-12,DET,IND,0.0871,11.482,1.095
1141,0022501112,2026-04-01,DEN,UTA,0.1415,7.066,1.165
1181,0022501152,2026-04-07,MIN,IND,0.1517,6.590,1.179
1207,0022501178,2026-04-10,MIN,HOU,0.5678,1.761,2.314
1196,0022501167,2026-04-09,IND,BKN,0.5717,1.749,2.335
1180,0022501151,2026-04-07,MIA,TOR,0.5773,1.732,2.366
1216,0022501187,2026-04-12,WAS,CLE,0.9169,1.091,12.034
1209,0022501180,2026-04-10,DAL,SAS,0.9400,1.064,16.669
1168,0022501139,2026-04-05,UTA,OKC,0.9582,1.044,23.925


## 11. Limitations

1. April is retrospective (viewed earlier in the project) — not a pristine holdout.
2. No bookmaker prices: no edge, CLV, or profitability claims.
3. 96 forecast games: wide intervals; the Elo-only baseline is statistically indistinguishable.
4. No injuries, lineups, or travel in the supplied data; late-season rotations unmodeled.

Fair odds above are zero-margin model prices, not recommended offer prices.